# REINFORCE in Python

Let's now implement the practical reward-to-go version of REINFORCE in Python. The implementation follows the previous section: it keeps the discounted reward-to-go target $G_t$ and drops the outer $\gamma^t$ factor from the policy-gradient term. We will test it on the problem of landing a space rocket booster on a floating platform. For this, we will use the `PlatformLander` environment.

In [ ]:
# Colab setup. Run this cell first.
!apt-get -qq update > /dev/null
!apt-get -qq install swig > /dev/null
!pip -q install platform_lander torch pandas matplotlib

In [ ]:
import csv
import math
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.distributions import Categorical

from platform_lander import PlatformLander
from platform_lander.platform_lander import BOOSTER_BOTTOM, BOOSTER_START_CLEARANCE, SCALE

RUNS_DIR = Path('/content/runs') if Path('/content').exists() else Path('runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# These match: gamma_dropped_rtg_reinforce.py --gamma 0.99 --learning-rate 1e-6
GAMMA = 0.99
LEARNING_RATE = 1e-6

# Script defaults. For a quick first Colab run, temporarily reduce EPISODES.
EPISODES = 150_000
MAX_STEPS = 400
HIDDEN_DIM = 64
SEED = 0
TARGET_WINDOW = 50
PRINT_EVERY = 250
ENABLE_WIND = False
WIND_POWER = 5.0

MODEL_FILE = RUNS_DIR / 'gamma_dropped_rtg_reinforce.pt'
CSV_FILE = RUNS_DIR / 'gamma_dropped_rtg_reinforce.csv'

## Platform Lander

The environment contains a reusable booster falling from space and a moving ocean platform where the booster should land. The booster has three controllable jets: one main jet at the bottom and two smaller attitude-control jets near the top, one on each side. Firing the bottom jet pushes the booster upward and helps slow its descent. Firing one of the top side jets creates a torque, rotating the booster clockwise or counterclockwise. The agent must learn when to fire these jets so that the booster descends slowly, stays upright, tracks the moving platform, and lands vertically.

![](https://raw.githubusercontent.com/aburkov/theDRLbook/main/test_environments/platform_lander/booster_landing_stages.png)

The platform floats on the ocean and moves horizontally from left to right and back again. If the booster lands upright on the platform with low enough speed, the episode is considered successful. If it misses the platform and falls into the ocean, flies out of bounds, hits the platform with its body, or touches the platform in a non-vertical position, the episode fails.

The environment also includes wind. When wind is enabled, it pushes the booster according to the wind direction and force, making the landing problem harder. The booster has a limited number of available jet fires, so the agent cannot simply keep firing forever; it must learn to use thrust efficiently while guiding the booster toward the moving platform.

Rewards in this environment are designed to encourage the booster to approach the platform, slow down, stay upright, and conserve fuel.

At each step, the environment computes a shaping score based on the current state. The score is better when the booster is close to the platform, moving slowly relative to it, and nearly vertical. Specifically, the shaping score subtracts $100\sqrt{x_{\text{rel}}^2 + y_{\text{rel}}^2}$ for distance from the platform's landing point (where $x_{\text{rel}}$ and $y_{\text{rel}}$ are the booster's horizontal and vertical positions relative to the platform), subtracts $100\sqrt{v_x^2 + v_y^2}$ for speed (where $v_x$ and $v_y$ are the booster's horizontal and vertical velocities), and subtracts $120|\theta|$ for tilt away from vertical (where $\theta$ is the booster's angle). It also gives a bonus of $12$ when the left landing foot touches the platform and another bonus of $12$ when the right landing foot touches the platform.

The actual per-step reward is the change in this shaping score from the previous step. So the agent receives positive reward when it improves its situation, and negative reward when it gets farther from a good landing state.

There is also a small fuel penalty whenever a jet is fired. Firing the bottom jet subtracts $0.30$, and firing one of the top attitude-control jets subtracts $0.03$.

Finally, terminal rewards are assigned at the end of an episode:

- successful vertical landing: $+100$
- failure: $-100$

The agent's goal is to learn a policy that lands the booster on the platform despite wind and a limited number of available jet fires. The policy must learn to land the booster vertically on the platform.

## Training a Platform Lander with Reward-to-Go REINFORCE

The `PlatformLander` environment has a discrete action space with four actions:

```text
0: do nothing
1: fire the upper-left attitude jet
2: fire the bottom engine
3: fire the upper-right attitude jet
```

The observation is an 11-dimensional vector. It contains:

```text
0: booster x-position relative to the platform, normalized by half the viewport width
1: booster y-position relative to the target landing height, normalized by half the viewport height
2: booster x-velocity relative to the platform x-velocity, scaled by viewport width and FPS
...
10: fraction of jet fires remaining
```

In [ ]:
env = PlatformLander(enable_wind=ENABLE_WIND, wind_power=WIND_POWER, wind_direction=(1.0, 0.0))
observation, info = env.reset(seed=SEED)

print('observation_space:', env.observation_space)
print('action_space:', env.action_space)
print('initial observation shape:', observation.shape)
print('initial observation:', np.round(observation, 3))

env.close()

Now we define the policy neural network.

In the code below:

- Line ➊ uses **orthogonal initialization** for the linear layer's weights and sets its bias to zero. Orthogonal initialization starts each linear layer with weight directions that are well separated from one another, which is a common choice in on-policy reinforcement learning implementations.
- In line ➋, the network maps the observation vector to a hidden representation.
- In line ➌, it applies a second hidden layer, again using the usual $\sqrt{2}$ gain for a `Tanh` policy network.
- In line ➍, the network outputs one logit for each possible action, using a small gain of $0.01$ so the initial categorical policy is close to uniform.

In [ ]:
def init_layer(
    layer: nn.Linear,
    *,
    gain: float = math.sqrt(2.0),
    bias_const: float = 0.0,
) -> nn.Linear:
    torch.nn.init.orthogonal_(layer.weight, gain=gain)  # ➊
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class Policy(nn.Module):
    """Categorical policy with OpenAI-Baselines/SB3-style initialization."""

    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            init_layer(nn.Linear(obs_dim, hidden_dim), gain=math.sqrt(2.0)),  # ➋
            nn.Tanh(),
            init_layer(nn.Linear(hidden_dim, hidden_dim), gain=math.sqrt(2.0)),  # ➌
            nn.Tanh(),
            init_layer(nn.Linear(hidden_dim, action_dim), gain=0.01),  # ➍
        )

    def forward(self, observation: torch.Tensor) -> torch.Tensor:
        return self.net(observation)

The policy does not output an action directly. It outputs logits. We convert these logits into a categorical distribution and sample from it.

In the code below:

- In line ➊, we convert the environment observation into a PyTorch tensor.
- In line ➋, the policy network produces logits.
- In line ➌, `Categorical` turns the logits into a probability distribution over the four actions.
- In line ➍, we sample an action from that distribution.
- In line ➎, we compute $\log \pi_\mathbf{\theta}(a_t \mid o_t)$ for the sampled action.

In [ ]:
def sample_action(policy: Policy, observation: np.ndarray, *, train: bool = True) -> tuple[int, torch.Tensor, torch.Tensor]:
    obs_tensor = torch.as_tensor(observation, dtype=torch.float32)  # ➊
    logits = policy(obs_tensor)  # ➋
    dist = Categorical(logits=logits)  # ➌

    if train:
        action_tensor = dist.sample()  # ➍
    else:
        action_tensor = torch.argmax(logits)

    action = int(action_tensor.item())
    log_prob = dist.log_prob(action_tensor)  # ➎
    return action, log_prob, obs_tensor


# Tiny check with an untrained policy.
env = PlatformLander()
observation, _ = env.reset(seed=SEED)
policy = Policy(obs_dim=int(env.observation_space.shape[0]), action_dim=int(env.action_space.n), hidden_dim=HIDDEN_DIM)
action, log_prob, obs_tensor = sample_action(policy, observation)
print('sampled action:', action)
print('log probability:', float(log_prob.detach()))
env.close()

During one episode, we store all observations, log-probabilities, and rewards.

In the code below:

- In line ➊, we store the observation tensor.
- In line ➋, we store the log-probability of the action actually taken.
- In line ➌, the environment applies the action to the booster physics.
- In line ➍, we store the reward returned by the environment.

The training script starts every episode from a randomized state: the booster position, platform position, platform direction, booster rotation, velocity, and angular velocity are all randomized. This prevents the policy from learning only one fixed starting condition.

In [ ]:
def randomize_start(env: PlatformLander, rng: np.random.Generator) -> np.ndarray:
    """Randomize platform and booster start, then return the fresh observation."""
    assert env.booster is not None
    assert env.platform is not None

    w = 600 / 30.0
    h = 400 / 30.0

    env.platform.position = (
        float(rng.uniform(env.platform_min_x, env.platform_max_x)),
        env.platform_y,
    )
    env.platform_direction = int(rng.choice([-1, 1]))
    env.platform.linearVelocity = (
        env.platform_speed * env.platform_direction,
        0.0,
    )

    env.booster.position = (
        float(rng.uniform(0.18 * w, 0.82 * w)),
        float(h + BOOSTER_BOTTOM / SCALE + rng.uniform(BOOSTER_START_CLEARANCE, 0.18)),
    )
    env.booster.angle = float(rng.uniform(-0.45, 0.45))
    env.booster.linearVelocity = (
        float(rng.uniform(-1.0, 1.0)),
        float(rng.uniform(-0.6, 0.4)),
    )
    env.booster.angularVelocity = float(rng.uniform(-0.6, 0.6))
    env.booster.awake = True

    env.ocean_contact = False
    env.platform_contact = False
    env.body_platform_contact = False
    env.left_foot_contact = False
    env.right_foot_contact = False
    env.failure_reason = None
    env.prev_shaping = None
    env.bottom_flame_power = 0.0
    env.top_flame_power = 0.0
    env.top_flame_direction = 0
    env.jet_fires_used = 0
    return env._get_state()


def discounted_return(rewards: list[float], gamma: float) -> float:
    total = 0.0
    discount = 1.0
    for reward in rewards:
        total += discount * reward
        discount *= gamma
    return total


def run_episode_data(
    env: PlatformLander,
    policy: Policy,
    rng: np.random.Generator,
    *,
    gamma: float,
    max_steps: int,
    train: bool,
) -> tuple[list[torch.Tensor], list[torch.Tensor], list[float], float, int, dict]:
    observation, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
    observation = randomize_start(env, rng)

    observations: list[torch.Tensor] = []
    log_probs: list[torch.Tensor] = []
    rewards: list[float] = []
    info: dict = {}

    for step in range(max_steps):
        action, log_prob, obs_tensor = sample_action(policy, observation, train=train)

        observations.append(obs_tensor)  # ➊
        log_probs.append(log_prob)  # ➋

        observation, reward, terminated, truncated, info = env.step(action)  # ➌
        rewards.append(float(reward))  # ➍

        if terminated or truncated:
            break

    if not terminated and not truncated:
        rewards[-1] = -100.0
        info = {
            **info,
            'success': False,
            'failure_reason': 'timeout',
        }

    episode_return = discounted_return(rewards, gamma)
    return observations, log_probs, rewards, episode_return, step + 1, info

After the episode ends, we compute a reward-to-go value for every timestep.

In line ➊, we walk backward through the episode and apply the recursion $G_t=r_{t+1}+\gamma G_{t+1}$. In line ➋, we reverse the list so `returns[t]` corresponds to the same timestep as `log_probs[t]`.

This implements:

$$
G_t=\sum_{k=t}^{T-1}\gamma^{k-t}r_{k+1}.
$$

The attached implementation deliberately stops here: it uses these discounted reward-to-go values directly and does not create a separate vector of outer $\gamma^t$ factors.

In [ ]:
def rewards_to_go(rewards: list[float], gamma: float) -> torch.Tensor:
    returns = []
    running_return = 0.0

    for reward in reversed(rewards):
        running_return = reward + gamma * running_return  # ➊
        returns.append(running_return)

    returns.reverse()  # ➋
    return torch.as_tensor(returns, dtype=torch.float32)

Now we can form the reward-to-go REINFORCE loss.

In the code below:

- In line ➊, `rtg[t]` stores $G_t$, the reward-to-go from timestep $t$.
- In line ➋, we turn the collected log-probabilities into a vector.
- In line ➌, we form $-\sum_{t=0}^{T-1}\log\pi_\mathbf{\theta}(a_t\mid o_t)G_t$, with no outer $\gamma^t$ factor.

The reason for the minus sign is that PyTorch's optimizers are designed to minimize a loss function by following the negative gradient. This is a typical scenario in supervised learning when we train neural networks to minimize the loss of making wrong prediction. In reinforcement learning, however, our goal is to maximize the expected total reward $J(\mathbf{\theta})$, not minimize it.

To apply the practical update using a minimizer, we minimize its negative. The exact gradient of the discounted objective would include the outer $\gamma^t$ factor derived earlier, but this implementation follows the gamma-dropped update described in the previous section. We could implement our own gradient ascent optimizer, but it is much simpler to reuse PyTorch's well-tested optimizers by negating the quantity whose gradient we want to ascend. Therefore, we define our policy loss as $-\sum_{t=0}^{T-1}\log\pi_\mathbf{\theta}(a_t\mid o_t)G_t$.

So minimizing this loss is equivalent to applying the update:

$$
\mathbf{\theta}
\leftarrow
\mathbf{\theta}
+
\alpha
\sum_{t=0}^{T-1}
\nabla_\mathbf{\theta}
\log \pi_\mathbf{\theta}(a_t \mid o_t)
G_t.
$$

In [ ]:
def reward_to_go_loss(
    log_probs: list[torch.Tensor],
    rewards: list[float],
    *,
    gamma: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    rtg = rewards_to_go(rewards, gamma)  # ➊
    log_prob_tensor = torch.stack(log_probs).reshape(-1)  # ➋

    rtg = rtg.to(device=log_prob_tensor.device, dtype=log_prob_tensor.dtype)

    assert log_prob_tensor.shape == rtg.shape

    policy_loss = -(log_prob_tensor * rtg).sum()  # ➌
    loss = policy_loss
    return loss, policy_loss

Finally, we perform the optimization step.

In the code below:

- In line ➊, we clear gradients from the previous episode.
- In line ➋, PyTorch computes the gradient of the loss.
- In line ➌, stochastic gradient descent updates the policy parameters.

In [ ]:
def save_training_csv(rows: list[dict[str, object]], csv_file: Path) -> Path:
    csv_file.parent.mkdir(parents=True, exist_ok=True)
    fieldnames: list[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    with csv_file.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return csv_file


def save_policy(policy: Policy, model_file: Path, *, obs_dim: int, action_dim: int) -> Path:
    model_file.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'policy_state_dict': policy.state_dict(),
            'obs_dim': obs_dim,
            'action_dim': action_dim,
            'hidden_dim': HIDDEN_DIM,
            'gamma': GAMMA,
            'learning_rate': LEARNING_RATE,
        },
        model_file,
    )
    return model_file


def train() -> tuple[Policy, pd.DataFrame]:
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)

    env = PlatformLander(
        enable_wind=ENABLE_WIND,
        wind_power=WIND_POWER,
        wind_direction=(1.0, 0.0),
    )

    policy = Policy(
        obs_dim=int(env.observation_space.shape[0]),
        action_dim=int(env.action_space.n),
        hidden_dim=HIDDEN_DIM,
    )
    obs_dim = int(env.observation_space.shape[0])
    action_dim = int(env.action_space.n)

    optimizer = torch.optim.SGD(policy.parameters(), lr=LEARNING_RATE)

    recent_returns: deque[float] = deque(maxlen=TARGET_WINDOW)
    recent_successes: deque[bool] = deque(maxlen=TARGET_WINDOW)
    recent_jet_fires: deque[int] = deque(maxlen=TARGET_WINDOW)
    training_rows: list[dict[str, object]] = []

    print(
        f'training_start script=gamma_dropped_rtg_reinforce '
        f'episodes={EPISODES} max_steps={MAX_STEPS} seed={SEED} '
        f'gamma={GAMMA} learning_rate={LEARNING_RATE}'
    )

    try:
        for episode in range(1, EPISODES + 1):
            observations, log_probs, rewards, episode_return, steps, info = run_episode_data(
                env,
                policy,
                rng,
                gamma=GAMMA,
                max_steps=MAX_STEPS,
                train=True,
            )

            loss, policy_loss = reward_to_go_loss(
                log_probs,
                rewards,
                gamma=GAMMA,
            )

            optimizer.zero_grad()  # ➊
            loss.backward()  # ➋
            optimizer.step()  # ➌

            recent_returns.append(episode_return)
            recent_successes.append(bool(info.get('success', False)))
            recent_jet_fires.append(int(info.get('jet_fires_used', 0)))

            average_return = float(np.mean(recent_returns))
            success_count = int(sum(recent_successes))
            average_jet_fires = float(np.mean(recent_jet_fires))

            training_rows.append(
                {
                    'episode': episode,
                    'return': episode_return,
                    'average_return': average_return,
                    'policy_loss': float(policy_loss.detach().item()),
                    'loss': float(loss.detach().item()),
                    'success_count': success_count,
                    'success_rate': success_count / len(recent_successes),
                    'jet_fires': int(info.get('jet_fires_used', 0)),
                    'average_jet_fires': average_jet_fires,
                    'steps': steps,
                    'success': bool(info.get('success', False)),
                    'failure_reason': info.get('failure_reason'),
                }
            )

            if episode == 1 or episode % PRINT_EVERY == 0:
                print(
                    f'episode={episode:5d} '
                    f'return={episode_return:8.2f} '
                    f'avg{len(recent_returns):02d}={average_return:8.2f} '
                    f'success{len(recent_successes):02d}={success_count:2d} '
                    f'fires={info.get("jet_fires_used", 0):3d} '
                    f'avgfires{len(recent_jet_fires):02d}={average_jet_fires:6.1f} '
                    f'steps={steps:4d} '
                    f'failure={info.get("failure_reason")}'
                )
    finally:
        env.close()

    model_path = save_policy(policy, MODEL_FILE, obs_dim=obs_dim, action_dim=action_dim)
    csv_path = save_training_csv(training_rows, CSV_FILE)
    print(f'saved_model={model_path}')
    print(f'saved_csv={csv_path}')

    return policy, pd.DataFrame(training_rows)

Run the training cell below. The default episode count matches the script, so it is intentionally long. For a quick classroom demonstration, change `EPISODES` near the top of the notebook to a smaller value such as `200` or `1_000`, then run the training cell.

In [ ]:
policy, history = train()
history.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history['episode'], history['return'], alpha=0.35, label='return')
axes[0].plot(history['episode'], history['average_return'], label=f'avg{TARGET_WINDOW}')
axes[0].set_title('Return')
axes[0].set_xlabel('Episode')
axes[0].legend()

axes[1].plot(history['episode'], history['success_rate'])
axes[1].set_title(f'Success rate over last {TARGET_WINDOW}')
axes[1].set_xlabel('Episode')
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

After training, we can run the learned policy greedily and render one RGB frame. In Colab, rendering to a Pygame window is not convenient, so this cell uses `render_mode="rgb_array"` and displays the final frame with Matplotlib.

In [ ]:
def evaluate_policy(policy: Policy, *, seed: int = SEED + 10_000, max_steps: int = MAX_STEPS) -> tuple[list[np.ndarray], list[float], float, dict]:
    rng = np.random.default_rng(seed)
    env = PlatformLander(
        render_mode='rgb_array',
        enable_wind=ENABLE_WIND,
        wind_power=WIND_POWER,
        wind_direction=(1.0, 0.0),
    )

    frames: list[np.ndarray] = []
    rewards: list[float] = []
    info: dict = {}

    try:
        observation, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
        observation = randomize_start(env, rng)

        for _ in range(max_steps):
            action, _, _ = sample_action(policy, observation, train=False)
            observation, reward, terminated, truncated, info = env.step(action)
            rewards.append(float(reward))
            frames.append(env.render())
            if terminated or truncated:
                break
    finally:
        env.close()

    return frames, rewards, discounted_return(rewards, GAMMA), info


frames, rewards, episode_return, info = evaluate_policy(policy)
print(f'evaluation_return={episode_return:.2f} steps={len(rewards)} info={info}')

if frames:
    plt.figure(figsize=(6, 4))
    plt.imshow(frames[-1])
    plt.axis('off')
    plt.show()